In [1]:
from __future__ import annotations
import itertools, math, re
from pathlib import Path
import numpy as np
import pandas as pd

INPUT_DIR = Path(r"C:\Users\firep\Downloads\simple_analysis_script\simple_analysis_script\sipm")
OUTPUT_DIR = Path(r"C:\Users\firep\Downloads\simple_analysis_script\timing_results_v2")

CHANNELS = ("C1", "C2", "C3", "C4")
ZOOM_TRACES = ("Z1", "Z2", "Z3", "Z4")
REFERENCE_CHANNEL = "C1"

SEARCH_START_NS, SEARCH_END_NS = -20.0, 20.0
MAX_FALL_SEARCH_NS = 50.0
REQUIRE_CFD_IN_WINDOW = True
PILEUP_REJECT_RATIO = 1.0

BASELINE_START_NS, BASELINE_END_NS = -50.0, -5.0
MIN_BASELINE_POINTS = 40
MIN_SNR = 5.0
LEADING_EDGE_SIGMA = 5.0
CFD_FRACTIONS = (0.10, 0.20, 0.50, 0.80, 0.90)

COINCIDENCE_WINDOW_NS = 10.0
OUTLIER_SIGMA = 5.0
POISSON_KMAX_CAP = 500
GAUSSIAN_FWHM = 2.354820045

COINCIDENCE_WINDOWS_NS = (0.05, 0.10, 0.20, 0.50, 1.0, 2.0, 5.0, 10.0)
N_MIXING_TRIALS = 200
MIXING_SEED = 20260730

EXCLUDE_FROM_TIMING = ("sensor_brown_ch_J15",)

PULSE_AMPLITUDE_MIN_SIGMA = 3.0
PE_HIST_MAX_V = 0.06
PE_HIST_BINS = 600
PE_SMOOTH_BINS = 9
PE_MIN_PEAK_V = 0.002

NOISE_HIST_RANGE_V = 0.05
NOISE_HIST_BINS = 2001
FIT_RANGE_SIGMA = 3.0
DELTA_T_FIT_BINS = 100

RUN_RAW_BACKGROUND_SCAN = True
PRETRIGGER_START_NS, PRETRIGGER_END_NS = -2000.0, -50.0
SCAN_SIGMA_THRESHOLDS = (3.0, 4.0, 5.0, 6.0)
PULSE_COUNT_SIGMA = 5.0
MAX_FILES_PER_CHANNEL = None
ACCIDENTAL_RESOLVING_TIME_NS = 1.0

PROGRESS_EVERY = 250

FILENAME_RE = re.compile(r"^(?P<channel>[^-]+)--(?P<device>.+?)--(?P<event>\d+)(?:\((?P<copy>\d+)\))?\.csv$", re.IGNORECASE)
TIMESTAMP_RE = re.compile(r"(\d{1,2}-[A-Za-z]{3}-\d{4}\s+\d{1,2}:\d{2}:\d{2}(?:\.\d+)?)")
TRAPEZOID = getattr(np, "trapezoid", np.trapz)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
written = {}

def write(name, frame):
    frame.to_csv(OUTPUT_DIR / name, index=False)
    written[name] = len(frame)
    print(f"  wrote {name} ({len(frame)} rows)")

In [2]:
def robust_sigma(values):
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    if v.size == 0:
        return math.nan
    s = 1.4826 * np.median(np.abs(v - np.median(v)))
    if s == 0.0 and v.size > 1:
        s = float(np.std(v, ddof=1))
    return float(s)

def crossing(t, y, level, peak, edge, lo, hi):
    if not (lo <= peak <= hi):
        return math.nan
    if edge == "rising":
        seg = y[lo:peak + 1]
        if seg.size < 2:
            return math.nan
        hits = np.flatnonzero((seg[:-1] < level) & (seg[1:] >= level))
        if hits.size == 0:
            return math.nan
        i = int(lo + hits[-1])
    else:
        seg = y[peak:hi + 1]
        if seg.size < 2:
            return math.nan
        hits = np.flatnonzero((seg[:-1] >= level) & (seg[1:] < level))
        if hits.size == 0:
            return math.nan
        i = int(peak + hits[0])
    y0, y1, t0, t1 = y[i], y[i + 1], t[i], t[i + 1]
    return float(t0) if y1 == y0 else float(t0 + (level - y0) * (t1 - t0) / (y1 - y0))

def excursion_peaks(y, threshold):
    if not np.isfinite(threshold) or threshold <= 0:
        return np.empty(0)
    above = y > threshold
    if not above.any():
        return np.empty(0)
    starts = np.flatnonzero((~above[:-1]) & above[1:]) + 1
    ends = np.flatnonzero(above[:-1] & (~above[1:])) + 1
    if above[0]:
        starts = np.r_[0, starts]
    if above[-1]:
        ends = np.r_[ends, len(y)]
    return np.array([float(y[a:b].max()) for a, b in zip(starts, ends)])

def n_excursions(y, threshold):
    if not np.isfinite(threshold) or threshold <= 0:
        return 0
    above = y > threshold
    if above.size == 0:
        return 0
    return int(np.count_nonzero(np.diff(above.astype(np.int8)) == 1)) + (1 if above[0] else 0)

try:
    from scipy.optimize import curve_fit
    HAVE_SCIPY = True
except ImportError:
    HAVE_SCIPY = False

def gaussian(x, amplitude, mu, sigma):
    return amplitude * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

def fit_gaussian(centers, counts, fit_range_sigma=FIT_RANGE_SIGMA):
    centers = np.asarray(centers, float)
    counts = np.asarray(counts, float)
    total = counts.sum()
    if total < 50:
        return {"fit_ok": False, "fit_method": "insufficient_counts"}
    mu0 = float(np.sum(centers * counts) / total)
    sigma0 = float(np.sqrt(max(np.sum(counts * (centers - mu0) ** 2) / total, 1e-30)))
    core = np.abs(centers - mu0) <= fit_range_sigma * sigma0
    x, y = centers[core], counts[core]
    ok = y > 0
    if ok.sum() < 5:
        return {"fit_ok": False, "fit_method": "too_few_bins"}
    if HAVE_SCIPY:
        try:
            popt, _ = curve_fit(gaussian, x, y, p0=[float(y.max()), mu0, sigma0], maxfev=20000)
            amplitude, mu, sigma = float(popt[0]), float(popt[1]), abs(float(popt[2]))
            method = "curve_fit"
        except Exception:
            amplitude, mu, sigma, method = float(y.max()), mu0, sigma0, "moments_fallback"
    else:
        coeffs = np.polyfit(x[ok], np.log(y[ok]), 2, w=np.sqrt(y[ok]))
        if coeffs[0] >= 0:
            amplitude, mu, sigma, method = float(y.max()), mu0, sigma0, "moments_fallback"
        else:
            sigma = float(np.sqrt(-1.0 / (2.0 * coeffs[0])))
            mu = float(-coeffs[1] / (2.0 * coeffs[0]))
            amplitude = float(np.exp(coeffs[2] - coeffs[1] ** 2 / (4.0 * coeffs[0])))
            method = "log_parabola"
    predicted = gaussian(x, amplitude, mu, sigma)
    residual = y - predicted
    variance = np.where(y > 0, y, 1.0)
    ndf = max(1, ok.sum() - 3)
    result = {"fit_ok": True, "fit_method": method, "fit_amplitude": amplitude,
              "fit_mu": mu, "fit_sigma": sigma,
              "fit_chi2_per_ndf": float(np.sum(residual ** 2 / variance) / ndf),
              "fit_ndf": int(ndf), "n_samples": float(total),
              "moment_sigma": sigma0, "moment_mu": mu0}
    with np.errstate(invalid="ignore"):
        z = (centers - mu) / sigma
    for k in (3.0, 4.0, 5.0, 6.0):
        measured = float(counts[np.abs(z) > k].sum() / total)
        expected = float(math.erfc(k / math.sqrt(2.0)))
        result[f"tail_fraction_beyond_{k:g}sigma"] = measured
        result[f"gaussian_expected_beyond_{k:g}sigma"] = expected
        result[f"tail_excess_ratio_{k:g}sigma"] = measured / expected if expected > 0 else math.nan
    return result

def extract(path, meta):
    trigger_text = ""
    with path.open("r", encoding="utf-8", errors="replace") as fh:
        header = [fh.readline().rstrip("\n") for _ in range(5)]
    if len(header) >= 4 and header[3].startswith("#"):
        trigger_text = header[3]

    frame = pd.read_csv(path, header=4, usecols=["Time", "Ampl"]).apply(pd.to_numeric, errors="coerce").dropna()
    if len(frame) < 3:
        raise ValueError("fewer than three valid samples")
    t = frame["Time"].to_numpy(float)
    a = frame["Ampl"].to_numpy(float)
    order = np.argsort(t)
    t, a = t[order], a[order]
    if np.any(np.diff(t) <= 0):
        t, idx = np.unique(t, return_index=True)
        a = a[idx]

    base_mask = (t >= BASELINE_START_NS * 1e-9) & (t <= BASELINE_END_NS * 1e-9)
    if np.count_nonzero(base_mask) >= MIN_BASELINE_POINTS:
        base_samples, base_method = a[base_mask], "time_window"
    else:
        n = min(max(MIN_BASELINE_POINTS, int(len(a) * 0.2)), max(3, len(a) // 2))
        base_samples, base_method = a[:n], "initial_fraction"

    baseline = float(np.median(base_samples))
    noise = robust_sigma(base_samples)
    centered = a - baseline

    win = (t >= SEARCH_START_NS * 1e-9) & (t <= SEARCH_END_NS * 1e-9)
    window_fallback = np.count_nonzero(win) < 3
    if window_fallback:
        win = np.ones_like(t, dtype=bool)
    lo = int(np.flatnonzero(win)[0])

    polarity = 1 if np.max(centered[win]) >= -np.min(centered[win]) else -1
    y = polarity * centered
    peak = int(lo + np.argmax(y[win]))
    amplitude = float(y[peak])
    snr = amplitude / noise if noise > 0 else math.inf

    global_peak = int(np.argmax(y))
    global_amplitude = float(y[global_peak])
    peak_outside = bool(global_peak != peak)
    ratio = global_amplitude / amplitude if amplitude > 0 else math.inf

    hi = int(np.searchsorted(t, t[peak] + MAX_FALL_SEARCH_NS * 1e-9, side="right") - 1)
    hi = max(peak, min(len(t) - 1, hi))

    pre = t < BASELINE_END_NS * 1e-9
    pre_sigma = float(np.max(y[pre]) / noise) if np.count_nonzero(pre) and noise > 0 else math.nan
    saturated = bool(np.count_nonzero(np.isclose(a, a[peak], rtol=0.0, atol=1e-12)) >= 5)

    row = {
        **meta,
        "file": path.name, "path": str(path.resolve()), "trigger_metadata": trigger_text,
        "n_samples": len(t), "dt_ps": float(np.median(np.diff(t)) * 1e12),
        "record_start_ns": float(t[0] * 1e9), "record_end_ns": float(t[-1] * 1e9),
        "baseline_method": base_method, "baseline_v": baseline, "noise_rms_robust_v": noise,
        "polarity": "positive" if polarity > 0 else "negative",
        "amplitude_v": amplitude, "snr": snr, "t_peak_ns": float(t[peak] * 1e9),
        "global_amplitude_v": global_amplitude, "t_global_peak_ns": float(t[global_peak] * 1e9),
        "peak_outside_window": peak_outside, "out_of_window_amplitude_ratio": ratio,
        "n_pulse_candidates": n_excursions(y, PULSE_COUNT_SIGMA * noise),
        "pretrigger_max_sigma": pre_sigma, "saturated": saturated,
        "search_window_fallback": bool(window_fallback),
    }

    for f in CFD_FRACTIONS:
        label = int(round(100 * f))
        row[f"t_cfd_{label}_ns"] = crossing(t, y, f * amplitude, peak, "rising", lo, hi) * 1e9
        row[f"t_fall_{label}_ns"] = crossing(t, y, f * amplitude, peak, "falling", lo, hi) * 1e9

    row["leading_edge_threshold_v"] = LEADING_EDGE_SIGMA * noise
    row["t_leading_edge_ns"] = crossing(t, y, LEADING_EDGE_SIGMA * noise, peak, "rising", lo, hi) * 1e9

    t10, t50, t90 = row["t_cfd_10_ns"], row["t_cfd_50_ns"], row["t_cfd_90_ns"]
    f10, f50, f90 = row["t_fall_10_ns"], row["t_fall_50_ns"], row["t_fall_90_ns"]
    row["rise_10_90_ns"] = t90 - t10 if np.isfinite(t10) and np.isfinite(t90) else math.nan
    row["fall_90_10_ns"] = f10 - f90 if np.isfinite(f90) and np.isfinite(f10) else math.nan
    row["fwhm_ns"] = f50 - t50 if np.isfinite(t50) and np.isfinite(f50) else math.nan

    if np.isfinite(t10) and np.isfinite(f10) and f10 > t10:
        m = (t >= t10 * 1e-9) & (t <= f10 * 1e-9)
        row["pulse_area_v_ns"] = float(TRAPEZOID(y[m], t[m]) * 1e9) if np.count_nonzero(m) >= 2 else math.nan
    else:
        row["pulse_area_v_ns"] = math.nan

    bad = []
    if window_fallback:
        bad.append("search_window_outside_record")
    if not (np.isfinite(snr) and snr >= MIN_SNR):
        bad.append("low_snr")
    if not amplitude > 0:
        bad.append("no_pulse")
    if saturated:
        bad.append("saturated")
    if not np.isfinite(t50):
        bad.append("no_cfd50_crossing")
    elif REQUIRE_CFD_IN_WINDOW and not (SEARCH_START_NS <= t50 <= SEARCH_END_NS):
        bad.append("cfd50_outside_window")
    if not np.isfinite(row["rise_10_90_ns"]):
        bad.append("no_rise_time")
    if PILEUP_REJECT_RATIO is not None and peak_outside and np.isfinite(ratio) and ratio > PILEUP_REJECT_RATIO:
        bad.append("larger_pulse_outside_window")

    row["timing_valid"] = not bad
    row["quality_flag"] = "ok" if not bad else ";".join(bad)
    if bad:
        for f in CFD_FRACTIONS:
            label = int(round(100 * f))
            row[f"t_cfd_{label}_ns"] = math.nan
            row[f"t_fall_{label}_ns"] = math.nan
        row["t_leading_edge_ns"] = math.nan
    return row

In [3]:
csv_files = sorted(p for p in INPUT_DIR.glob("*.csv") if p.is_file())
if not csv_files:
    raise FileNotFoundError(f"No CSV files in {INPUT_DIR}")
print(f"Found {len(csv_files)} files")

records, failures, skipped = [], [], []
for i, path in enumerate(csv_files, start=1):
    m = FILENAME_RE.match(path.name)
    if m is None:
        skipped.append(path.name)
        continue
    meta = {"channel": m.group("channel"), "device": m.group("device"),
            "event": int(m.group("event")), "copy_index": int(m.group("copy") or 0)}
    try:
        records.append(extract(path, meta))
    except Exception as exc:
        failures.append({"file": path.name, "error": str(exc)})
    if i % PROGRESS_EVERY == 0 or i == len(csv_files):
        print(f"  extracted {i}/{len(csv_files)}")

if not records:
    raise RuntimeError("No waveforms analyzed successfully")

features = pd.DataFrame(records).sort_values(["channel", "device", "event", "copy_index"]).reset_index(drop=True)
features.insert(2, "trace_role", np.where(features["channel"].isin(CHANNELS), "physical",
                np.where(features["channel"].isin(ZOOM_TRACES), "zoom", "other")))
features["trigger_timestamp"] = pd.to_datetime(
    features["trigger_metadata"].str.extract(TIMESTAMP_RE, expand=False), errors="coerce")

write("waveform_features.csv", features)
if failures:
    write("failed_files.csv", pd.DataFrame(failures))
if skipped:
    write("skipped_filenames.csv", pd.DataFrame({"skipped_filename": skipped}))

physical_all = features[features["trace_role"] == "physical"].copy()
physical = physical_all[~physical_all["device"].isin(EXCLUDE_FROM_TIMING)].copy()
print(f"Physical traces: {len(physical_all)}   used for timing: {len(physical)}   "
      f"valid: {int(physical['timing_valid'].sum())}")

Found 20688 files
  extracted 250/20688
  extracted 500/20688
  extracted 750/20688
  extracted 1000/20688
  extracted 1250/20688
  extracted 1500/20688
  extracted 1750/20688
  extracted 2000/20688
  extracted 2250/20688
  extracted 2500/20688
  extracted 2750/20688
  extracted 3000/20688
  extracted 3250/20688
  extracted 3500/20688
  extracted 3750/20688
  extracted 4000/20688
  extracted 4250/20688
  extracted 4500/20688
  extracted 4750/20688
  extracted 5000/20688
  extracted 5250/20688
  extracted 5500/20688
  extracted 5750/20688
  extracted 6000/20688
  extracted 6250/20688
  extracted 6500/20688
  extracted 6750/20688
  extracted 7000/20688
  extracted 7250/20688
  extracted 7500/20688
  extracted 7750/20688
  extracted 8000/20688
  extracted 8250/20688
  extracted 8500/20688
  extracted 8750/20688
  extracted 9000/20688
  extracted 9250/20688
  extracted 9500/20688
  extracted 9750/20688
  extracted 10000/20688
  extracted 10250/20688
  extracted 10500/20688
  extracted 1075

In [4]:
print("Deriving acquisition runs from trigger timestamps")
audit, runs = [], {}
for device, group in physical.groupby("device", dropna=False):
    missing = set(CHANNELS) - set(group["channel"].unique())
    if missing:
        audit.append({"device": device, "usable": False, "reason": f"missing {sorted(missing)}",
                      "n_events": group["event"].nunique(), "trigger_agreement": math.nan})
        continue
    pivot = group.pivot_table(index="event", columns="channel", values="trigger_metadata",
                              aggfunc="first").reindex(columns=list(CHANNELS)).dropna()
    agreement = float((pivot.nunique(axis=1) == 1).mean()) if len(pivot) else 0.0
    usable = agreement > 0.99
    audit.append({"device": device, "usable": usable,
                  "reason": "ok" if usable else "trigger timestamps disagree",
                  "n_events": int(len(pivot)), "trigger_agreement": agreement})
    if usable:
        runs[str(device)] = {c: str(device) for c in CHANNELS}

write("acquisition_run_audit.csv", pd.DataFrame(audit))
if not runs:
    raise RuntimeError("No device has four channels with consistent trigger timestamps")

stamps = (physical.dropna(subset=["trigger_timestamp"]).groupby("device")["trigger_timestamp"]
          .agg(["min", "max", "count"]).reset_index()
          .rename(columns={"min": "first_trigger", "max": "last_trigger", "count": "n_traces"}))
overlap = []
for a, b in itertools.combinations(list(stamps["device"]), 2):
    ra = stamps[stamps["device"] == a].iloc[0]
    rb = stamps[stamps["device"] == b].iloc[0]
    overlap.append({"device_a": a, "device_b": b,
                    "overlaps": bool((ra["first_trigger"] <= rb["last_trigger"]) and
                                     (rb["first_trigger"] <= ra["last_trigger"]))})
write("acquisition_timestamps.csv", stamps)
if overlap:
    write("acquisition_overlap.csv", pd.DataFrame(overlap))

Deriving acquisition runs from trigger timestamps
  wrote acquisition_run_audit.csv (2 rows)
  wrote acquisition_timestamps.csv (2 rows)
  wrote acquisition_overlap.csv (1 rows)


In [5]:
print("Building fourfold coincidences")
t_cols = [f"t_{c}_ns" for c in CHANNELS]
pairs = list(itertools.combinations(CHANNELS, 2))
status_frames, pair_frames, run_rows = [], [], []

for run, mapping in runs.items():
    tables = []
    for c in CHANNELS:
        sub = physical[(physical["channel"] == c) & (physical["device"] == mapping[c])].copy()
        sub["_v"] = sub["timing_valid"].fillna(False).astype(int)
        sub["_s"] = pd.to_numeric(sub["snr"], errors="coerce").fillna(-np.inf)
        sub = (sub.sort_values(["event", "_v", "_s", "copy_index"], ascending=[True, False, False, True])
               .drop_duplicates("event", keep="first"))
        table = sub[["event", "file", "timing_valid", "quality_flag", "t_cfd_50_ns", "amplitude_v",
                     "snr", "trigger_metadata", "rise_10_90_ns", "peak_outside_window"]].rename(
            columns={"file": f"file_{c}", "timing_valid": f"valid_{c}", "quality_flag": f"quality_{c}",
                     "t_cfd_50_ns": f"t_{c}_ns", "amplitude_v": f"amplitude_{c}_v", "snr": f"snr_{c}",
                     "trigger_metadata": f"trigger_{c}", "rise_10_90_ns": f"rise_{c}_ns",
                     "peak_outside_window": f"peak_outside_{c}"})
        table[f"present_{c}"] = True
        tables.append(table)

    status = tables[0]
    for table in tables[1:]:
        status = status.merge(table, on="event", how="outer")
    status.insert(0, "coincidence_run", run)
    status = status.sort_values("event").reset_index(drop=True)

    for c in CHANNELS:
        status[f"present_{c}"] = status[f"present_{c}"].eq(True)
        status[f"valid_{c}"] = status[f"valid_{c}"].eq(True)

    status["complete_fourfold"] = status[[f"present_{c}" for c in CHANNELS]].all(axis=1)
    status["all_timing_valid"] = (status["complete_fourfold"]
                                  & status[[f"valid_{c}" for c in CHANNELS]].all(axis=1)
                                  & status[t_cols].notna().all(axis=1))
    status["same_trigger"] = status[[f"trigger_{c}" for c in CHANNELS]].nunique(axis=1, dropna=False) == 1
    status["time_spread_ns"] = np.where(status["all_timing_valid"],
                                        status[t_cols].max(axis=1) - status[t_cols].min(axis=1), np.nan)
    status["within_window"] = status["all_timing_valid"] & (status["time_spread_ns"] <= COINCIDENCE_WINDOW_NS)
    status["accepted_fourfold"] = status["within_window"] & status["same_trigger"]
    for c in CHANNELS:
        status[f"delta_t_{c}_minus_{REFERENCE_CHANNEL}_ns"] = status[f"t_{c}_ns"] - status[f"t_{REFERENCE_CHANNEL}_ns"]

    accepted = status[status["accepted_fourfold"]]
    status_frames.append(status)

    for a, b in pairs:
        if accepted.empty:
            continue
        p = accepted[["coincidence_run", "event", f"t_{a}_ns", f"t_{b}_ns", f"amplitude_{a}_v",
                      f"amplitude_{b}_v", f"snr_{a}", f"snr_{b}"]].rename(
            columns={f"t_{a}_ns": "t_a_ns", f"t_{b}_ns": "t_b_ns", f"amplitude_{a}_v": "amplitude_a_v",
                     f"amplitude_{b}_v": "amplitude_b_v", f"snr_{a}": "snr_a", f"snr_{b}": "snr_b"}).copy()
        p.insert(2, "channel_a", a)
        p.insert(3, "channel_b", b)
        p["delta_t_ns"] = p["t_b_ns"] - p["t_a_ns"]
        pair_frames.append(p)

    run_rows.append({"coincidence_run": run,
                     "channel_device_mapping": "; ".join(f"{c}={mapping[c]}" for c in CHANNELS),
                     "coincidence_window_ns": COINCIDENCE_WINDOW_NS,
                     "n_candidate_events": len(status),
                     "n_complete_fourfold": int(status["complete_fourfold"].sum()),
                     "n_same_trigger": int(status["same_trigger"].sum()),
                     "n_all_timing_valid": int(status["all_timing_valid"].sum()),
                     "n_within_window": int(status["within_window"].sum()),
                     "n_accepted_fourfold": int(status["accepted_fourfold"].sum()),
                     "accepted_fraction": float(status["accepted_fourfold"].mean()),
                     "time_spread_median_ns": float(accepted["time_spread_ns"].median()) if len(accepted) else math.nan,
                     "time_spread_robust_sigma_ns": robust_sigma(accepted["time_spread_ns"]) if len(accepted) > 1 else math.nan,
                     "time_spread_max_ns": float(accepted["time_spread_ns"].max()) if len(accepted) else math.nan})

event_status = pd.concat(status_frames, ignore_index=True)
pairwise = pd.concat(pair_frames, ignore_index=True) if pair_frames else pd.DataFrame()

Building fourfold coincidences


In [6]:
print("Applying time-walk correction")
pairwise["delta_t_corrected_ns"] = pairwise["delta_t_ns"]
pairwise["walk_slope"] = np.nan
for _, group in pairwise.groupby(["coincidence_run", "channel_a", "channel_b"]):
    amp_a = pd.to_numeric(group["amplitude_a_v"], errors="coerce").to_numpy(float)
    amp_b = pd.to_numeric(group["amplitude_b_v"], errors="coerce").to_numpy(float)
    delta = pd.to_numeric(group["delta_t_ns"], errors="coerce").to_numpy(float)
    with np.errstate(divide="ignore", invalid="ignore"):
        x = 1.0 / np.sqrt(amp_b) - 1.0 / np.sqrt(amp_a)
    ok = np.isfinite(x) & np.isfinite(delta)
    if ok.sum() <= 10:
        continue
    sigma, median = robust_sigma(delta[ok]), np.median(delta[ok])
    core = ok & (np.abs(delta - median) <= OUTLIER_SIGMA * sigma)
    if core.sum() > 10 and np.ptp(x[core]) > 0:
        slope = np.polyfit(x[core], delta[core], 1)[0]
        pairwise.loc[group.index, "delta_t_corrected_ns"] = delta - slope * x
        pairwise.loc[group.index, "walk_slope"] = slope

write("coincidence_event_status.csv", event_status)
write("fourfold_coincidences.csv", event_status[event_status["accepted_fourfold"]])
write("coincidence_pairwise_deltas.csv", pairwise)
write("coincidence_summary.csv", pd.DataFrame(run_rows))

Applying time-walk correction
  wrote coincidence_event_status.csv (2584 rows)
  wrote fourfold_coincidences.csv (2074 rows)
  wrote coincidence_pairwise_deltas.csv (12444 rows)
  wrote coincidence_summary.csv (2 rows)


In [7]:
print("Summarizing pairwise timing")
summary_rows = []
for column in ("delta_t_ns", "delta_t_corrected_ns"):
    for (run, a, b), group in pairwise.groupby(["coincidence_run", "channel_a", "channel_b"]):
        v = pd.to_numeric(group[column], errors="coerce").dropna().to_numpy(float)
        if v.size == 0:
            continue
        median = float(np.median(v))
        rs = robust_sigma(v) if v.size > 1 else math.nan
        sd = float(np.std(v, ddof=1)) if v.size > 1 else math.nan
        out = np.abs(v - median) > OUTLIER_SIGMA * rs if np.isfinite(rs) and rs > 0 else np.zeros_like(v, bool)
        core = v[~out]
        summary_rows.append({
            "coincidence_run": run, "channel_a": a, "channel_b": b, "timing_column": column,
            "n_events": int(v.size), "n_outliers": int(out.sum()), "outlier_fraction": float(out.mean()),
            "mean_delta_t_ns": float(np.mean(v)), "median_delta_t_ns": median,
            "robust_sigma_ps": rs * 1e3, "std_all_ps": sd * 1e3,
            "std_core_ps": float(np.std(core, ddof=1)) * 1e3 if core.size > 1 else math.nan,
            "fwhm_from_robust_ps": GAUSSIAN_FWHM * rs * 1e3,
            "fwhm_from_std_all_ps": GAUSSIAN_FWHM * sd * 1e3,
            "p16_ps": float(np.percentile(v, 16)) * 1e3, "p84_ps": float(np.percentile(v, 84)) * 1e3})

pair_summary = pd.DataFrame(summary_rows)
write("coincidence_pairwise_summary.csv", pair_summary)

fit_rows, hist_rows = [], []
for column in ("delta_t_ns", "delta_t_corrected_ns"):
    for (run, a, b), group in pairwise.groupby(["coincidence_run", "channel_a", "channel_b"]):
        v = pd.to_numeric(group[column], errors="coerce").dropna().to_numpy(float)
        if v.size < 50:
            continue
        median = float(np.median(v))
        rs = robust_sigma(v)
        core = v[np.abs(v - median) <= 8 * max(rs, 1e-9)]
        if core.size < 50:
            continue
        counts, edges = np.histogram(core * 1e3, bins=DELTA_T_FIT_BINS)
        centers = 0.5 * (edges[:-1] + edges[1:])
        fit = fit_gaussian(centers, counts)
        fit_rows.append({"coincidence_run": run, "channel_a": a, "channel_b": b,
                         "timing_column": column, "n_events_total": int(v.size),
                         "n_events_core": int(core.size),
                         "robust_sigma_ps": rs * 1e3, "median_ps": median * 1e3,
                         "fit_sigma_ps": fit.get("fit_sigma", math.nan),
                         "fit_mu_ps": fit.get("fit_mu", math.nan),
                         "fit_fwhm_ps": GAUSSIAN_FWHM * fit.get("fit_sigma", math.nan),
                         **{k: fit[k] for k in fit if k not in ("fit_sigma", "fit_mu")}})
        if column == "delta_t_ns":
            hist_rows.append(pd.DataFrame({"coincidence_run": run, "channel_a": a, "channel_b": b,
                                           "bin_center_ps": centers, "counts": counts}))

write("pairwise_gaussian_fit.csv", pd.DataFrame(fit_rows))
if hist_rows:
    write("pairwise_delta_t_histogram.csv", pd.concat(hist_rows, ignore_index=True))

resolution_rows = []
for (run, column), group in pair_summary.groupby(["coincidence_run", "timing_column"]):
    lookup = {frozenset((r.channel_a, r.channel_b)): r.robust_sigma_ps
              for r in group.itertuples() if np.isfinite(r.robust_sigma_ps)}
    for c in CHANNELS:
        others = [x for x in CHANNELS if x != c]
        estimates, negatives = [], 0
        for b, d in itertools.combinations(others, 2):
            keys = (frozenset((c, b)), frozenset((c, d)), frozenset((b, d)))
            if not all(k in lookup for k in keys):
                continue
            var = (lookup[keys[0]] ** 2 + lookup[keys[1]] ** 2 - lookup[keys[2]] ** 2) / 2.0
            if var > 0:
                estimates.append(math.sqrt(var))
            else:
                negatives += 1
        resolution_rows.append({
            "coincidence_run": run, "channel": c, "timing_column": column,
            "n_triplet_estimates": len(estimates), "n_negative_solutions": negatives,
            "sigma_channel_ps": float(np.mean(estimates)) if estimates else math.nan,
            "sigma_channel_spread_ps": float(np.std(estimates, ddof=1)) if len(estimates) > 1 else math.nan,
            "fwhm_channel_ps": GAUSSIAN_FWHM * float(np.mean(estimates)) if estimates else math.nan})

write("channel_resolution_decomposition.csv", pd.DataFrame(resolution_rows))

offset_rows = []
for (run, b), group in pairwise[pairwise["channel_a"] == REFERENCE_CHANNEL].groupby(["coincidence_run", "channel_b"]):
    v = pd.to_numeric(group["delta_t_ns"], errors="coerce").dropna()
    offset_rows.append({"coincidence_run": run, "channel": b, "reference_channel": REFERENCE_CHANNEL,
                        "n_events": int(v.size), "median_offset_ps": float(v.median()) * 1e3,
                        "robust_sigma_ps": robust_sigma(v) * 1e3})
for run in pairwise["coincidence_run"].unique():
    offset_rows.append({"coincidence_run": run, "channel": REFERENCE_CHANNEL,
                        "reference_channel": REFERENCE_CHANNEL, "n_events": math.nan,
                        "median_offset_ps": 0.0, "robust_sigma_ps": math.nan})
write("channel_offsets.csv", pd.DataFrame(offset_rows).sort_values(["coincidence_run", "channel"]))

Summarizing pairwise timing
  wrote coincidence_pairwise_summary.csv (24 rows)
  wrote pairwise_gaussian_fit.csv (24 rows)
  wrote pairwise_delta_t_histogram.csv (1200 rows)
  wrote channel_resolution_decomposition.csv (16 rows)
  wrote channel_offsets.csv (8 rows)


In [8]:
print("Summarizing per-channel features and background levels")
agg_columns = ["amplitude_v", "noise_rms_robust_v", "snr", "t_cfd_50_ns", "rise_10_90_ns",
               "fwhm_ns", "pulse_area_v_ns", "baseline_v"]
rows = []
for (channel, device), group in physical.groupby(["channel", "device"], dropna=False):
    valid = group[group["timing_valid"]]
    row = {"channel": channel, "device": device, "n_total": len(group), "n_valid": len(valid),
           "valid_fraction": len(valid) / len(group) if len(group) else math.nan,
           "frac_peak_outside_window": float(group["peak_outside_window"].mean()),
           "frac_saturated": float(group["saturated"].mean()),
           "median_pulse_candidates": float(group["n_pulse_candidates"].median()),
           "mean_pulse_candidates": float(group["n_pulse_candidates"].mean()),
           "pretrigger_max_sigma_median": float(group["pretrigger_max_sigma"].median()),
           "pretrigger_max_sigma_p99": float(group["pretrigger_max_sigma"].quantile(0.99)),
           "low_snr_fraction": float((pd.to_numeric(group["snr"], errors="coerce") < MIN_SNR).mean()),
           "t_peak_p01_ns": float(group["t_peak_ns"].quantile(0.01)),
           "t_peak_p99_ns": float(group["t_peak_ns"].quantile(0.99))}
    for column in agg_columns:
        v = pd.to_numeric(valid[column], errors="coerce").dropna().to_numpy(float)
        row[f"{column}_mean"] = float(np.mean(v)) if v.size else math.nan
        row[f"{column}_median"] = float(np.median(v)) if v.size else math.nan
        row[f"{column}_std"] = float(np.std(v, ddof=1)) if v.size > 1 else math.nan
        row[f"{column}_robust_sigma"] = robust_sigma(v) if v.size > 1 else math.nan
    for column, label in (("baseline_v", "baseline"), ("noise_rms_robust_v", "noise")):
        v = pd.to_numeric(group[column], errors="coerce").dropna().to_numpy(float)
        s = robust_sigma(v)
        row[f"{label}_outlier_fraction_5sigma"] = (
            float(np.mean(np.abs(v - np.median(v)) > 5 * s)) if v.size > 1 and s > 0 else math.nan)
    rows.append(row)
write("physical_channel_summary.csv", pd.DataFrame(rows).sort_values(["device", "channel"]))

write("rejection_reasons.csv", physical.loc[~physical["timing_valid"], "quality_flag"]
      .value_counts().rename_axis("rejection_reason").reset_index(name="events"))

drift_rows = []
for (channel, device), group in physical.groupby(["channel", "device"]):
    ordered = group.sort_values("event")
    x = pd.to_numeric(ordered["event"], errors="coerce").to_numpy(float)
    for column in ("baseline_v", "noise_rms_robust_v"):
        y = pd.to_numeric(ordered[column], errors="coerce").to_numpy(float)
        ok = np.isfinite(x) & np.isfinite(y)
        slope = np.polyfit(x[ok], y[ok], 1)[0] if ok.sum() > 10 and np.ptp(x[ok]) > 0 else math.nan
        drift_rows.append({"channel": channel, "device": device, "quantity": column,
                           "slope_v_per_event": slope,
                           "total_drift_v": slope * np.ptp(x[ok]) if np.isfinite(slope) else math.nan,
                           "median_v": float(np.nanmedian(y)) if y.size else math.nan})
write("background_drift.csv", pd.DataFrame(drift_rows))

zoom = features[features["trace_role"] == "zoom"]
if not zoom.empty:
    write("zoom_trace_features.csv", zoom)

Summarizing per-channel features and background levels
  wrote physical_channel_summary.csv (8 rows)
  wrote rejection_reasons.csv (10 rows)
  wrote background_drift.csv (16 rows)
  wrote zoom_trace_features.csv (10344 rows)


In [9]:
print("Testing whether background counts are Poisson")
poisson_rows, poisson_dist_rows = [], []
for (channel, device), group in physical.groupby(["channel", "device"]):
    counts = pd.to_numeric(group["n_pulse_candidates"], errors="coerce").dropna().to_numpy(float)
    if counts.size < 20:
        continue
    mean, var = float(counts.mean()), float(counts.var(ddof=1))
    kmax = int(counts.max())
    k_grid = np.arange(0, min(kmax, POISSON_KMAX_CAP) + 1)
    observed = np.array([np.count_nonzero(counts == k) for k in k_grid], float)

    # Poisson pmf in log space: mean ** k overflows float64 for large k.
    if mean > 0:
        log_pmf = (-mean + k_grid * math.log(mean)
                   - np.array([math.lgamma(k + 1.0) for k in k_grid]))
        expected = counts.size * np.exp(np.clip(log_pmf, -700.0, 0.0))
        expected[log_pmf < -700.0] = 0.0
    else:
        expected = np.zeros_like(k_grid, dtype=float)
        expected[0] = float(counts.size)

    usable = expected >= 5
    chi2 = float(np.sum((observed[usable] - expected[usable]) ** 2 / expected[usable])) if usable.any() else math.nan
    ndf = max(1, int(usable.sum()) - 2)
    poisson_rows.append({"channel": channel, "device": device, "n_waveforms": int(counts.size),
                         "mean_pulses_per_record": mean, "variance_pulses_per_record": var,
                         "max_pulses_per_record": kmax,
                         "k_truncated": bool(kmax > POISSON_KMAX_CAP),
                         "fano_factor": var / mean if mean > 0 else math.nan,
                         "fano_uncertainty": math.sqrt(2.0 / (counts.size - 1)),
                         "chi2_poisson": chi2, "ndf": ndf,
                         "chi2_per_ndf": chi2 / ndf if np.isfinite(chi2) else math.nan})
    if kmax > POISSON_KMAX_CAP:
        print(f"  {device} {channel}: max {kmax} pulses per record, truncated at {POISSON_KMAX_CAP}")
    for i, k in enumerate(k_grid):
        poisson_dist_rows.append({"channel": channel, "device": device, "n_pulses": int(k),
                                  "observed": observed[i], "poisson_expected": expected[i]})

write("background_poisson_test.csv", pd.DataFrame(poisson_rows))
write("background_poisson_distribution.csv", pd.DataFrame(poisson_dist_rows))

Testing whether background counts are Poisson
  hcsipm C1: max 1228 pulses per record, truncated at 500
  sipm C1: max 937 pulses per record, truncated at 500
  hcsipm C2: max 951 pulses per record, truncated at 500
  sipm C2: max 695 pulses per record, truncated at 500
  hcsipm C3: max 882 pulses per record, truncated at 500
  sipm C3: max 781 pulses per record, truncated at 500
  hcsipm C4: max 1056 pulses per record, truncated at 500
  sipm C4: max 3145 pulses per record, truncated at 500
  wrote background_poisson_test.csv (8 rows)
  wrote background_poisson_distribution.csv (4008 rows)


In [10]:
print("Predicting timing resolution from noise and slew rate")
slew_rows = []
for (channel, device), group in physical.groupby(["channel", "device"]):
    good = group[group["timing_valid"].astype(bool)]
    amplitude = pd.to_numeric(good["amplitude_v"], errors="coerce").to_numpy(float)
    rise = pd.to_numeric(good["rise_10_90_ns"], errors="coerce").to_numpy(float)
    noise = pd.to_numeric(good["noise_rms_robust_v"], errors="coerce").to_numpy(float)
    ok = np.isfinite(amplitude) & np.isfinite(rise) & np.isfinite(noise) & (rise > 0)
    if ok.sum() < 10:
        continue
    slew = 0.8 * amplitude[ok] / rise[ok]
    jitter_ps = 1e3 * noise[ok] / slew
    slew_rows.append({"channel": channel, "device": device, "n_events": int(ok.sum()),
                      "median_amplitude_v": float(np.median(amplitude[ok])),
                      "median_rise_10_90_ns": float(np.median(rise[ok])),
                      "median_noise_v": float(np.median(noise[ok])),
                      "median_slew_v_per_ns": float(np.median(slew)),
                      "predicted_sigma_ps": float(np.median(jitter_ps)),
                      "predicted_sigma_spread_ps": robust_sigma(jitter_ps)})

slew = pd.DataFrame(slew_rows)
write("timing_noise_prediction.csv", slew)

compare_rows = []
predicted_lookup = {(r.device, r.channel): r.predicted_sigma_ps for r in slew.itertuples()}
for r in pair_summary[pair_summary["timing_column"] == "delta_t_ns"].itertuples():
    device = runs.get(r.coincidence_run, {}).get(r.channel_a)
    pa = predicted_lookup.get((device, r.channel_a), math.nan)
    pb = predicted_lookup.get((device, r.channel_b), math.nan)
    predicted_pair = math.sqrt(pa ** 2 + pb ** 2) if np.isfinite(pa) and np.isfinite(pb) else math.nan
    compare_rows.append({"coincidence_run": r.coincidence_run, "channel_a": r.channel_a,
                         "channel_b": r.channel_b,
                         "predicted_sigma_a_ps": pa, "predicted_sigma_b_ps": pb,
                         "predicted_pair_sigma_ps": predicted_pair,
                         "measured_pair_sigma_ps": r.robust_sigma_ps,
                         "measured_over_predicted": (r.robust_sigma_ps / predicted_pair
                                                     if np.isfinite(predicted_pair) and predicted_pair > 0
                                                     else math.nan),
                         "excess_sigma_ps": (math.sqrt(max(0.0, r.robust_sigma_ps ** 2 - predicted_pair ** 2))
                                             if np.isfinite(predicted_pair) else math.nan)})
write("timing_predicted_vs_measured.csv", pd.DataFrame(compare_rows))

Predicting timing resolution from noise and slew rate
  wrote timing_noise_prediction.csv (8 rows)
  wrote timing_predicted_vs_measured.csv (12 rows)


In [11]:
print("Estimating accidental coincidences by event mixing")
rng = np.random.default_rng(MIXING_SEED)
mix_rows, pair_mix_rows = [], []

for run, group in event_status.groupby("coincidence_run"):
    valid = group[group["all_timing_valid"].astype(bool)]
    times = valid[t_cols].to_numpy(float)
    times = times[np.isfinite(times).all(axis=1)]
    n = times.shape[0]
    if n < 10:
        continue
    aligned = times - np.median(times - times[:, [0]], axis=0)
    true_spread = aligned.max(axis=1) - aligned.min(axis=1)

    for window in COINCIDENCE_WINDOWS_NS:
        true_rate = float(np.mean(true_spread <= window))
        acc = np.empty(N_MIXING_TRIALS)
        for k in range(N_MIXING_TRIALS):
            mixed = aligned.copy()
            for col in range(1, mixed.shape[1]):
                mixed[:, col] = aligned[rng.permutation(n), col]
            s = mixed.max(axis=1) - mixed.min(axis=1)
            acc[k] = float(np.mean(s <= window))
        mix_rows.append({"coincidence_run": run, "coincidence_window_ns": window, "n_events": n,
                         "n_mixing_trials": N_MIXING_TRIALS, "true_coincidence_fraction": true_rate,
                         "accidental_fraction_mean": float(acc.mean()),
                         "accidental_fraction_std": float(acc.std(ddof=1)),
                         "signal_fraction": max(0.0, true_rate - float(acc.mean())),
                         "accidental_contamination": float(acc.mean()) / true_rate if true_rate > 0 else math.nan})

    for a, b in pairs:
        ta = pd.to_numeric(valid[f"t_{a}_ns"], errors="coerce").to_numpy(float)
        tb = pd.to_numeric(valid[f"t_{b}_ns"], errors="coerce").to_numpy(float)
        ok = np.isfinite(ta) & np.isfinite(tb)
        ta, tb = ta[ok], tb[ok]
        if ta.size < 10:
            continue
        shifted = tb - float(np.median(tb - ta))
        for window in COINCIDENCE_WINDOWS_NS:
            true_rate = float(np.mean(np.abs(shifted - ta) <= window))
            acc = np.empty(N_MIXING_TRIALS)
            for k in range(N_MIXING_TRIALS):
                acc[k] = float(np.mean(np.abs(shifted[rng.permutation(ta.size)] - ta) <= window))
            pair_mix_rows.append({"coincidence_run": run, "channel_a": a, "channel_b": b,
                                  "coincidence_window_ns": window, "n_events": int(ta.size),
                                  "true_coincidence_fraction": true_rate,
                                  "accidental_fraction_mean": float(acc.mean()),
                                  "accidental_fraction_std": float(acc.std(ddof=1)),
                                  "accidental_contamination": float(acc.mean()) / true_rate if true_rate > 0 else math.nan})
    print(f"  mixed {run}")

write("accidental_fourfold_mixing.csv", pd.DataFrame(mix_rows))
write("accidental_pairwise_mixing.csv", pd.DataFrame(pair_mix_rows))

Estimating accidental coincidences by event mixing
  mixed hcsipm
  mixed sipm
  wrote accidental_fourfold_mixing.csv (16 rows)
  wrote accidental_pairwise_mixing.csv (96 rows)


In [12]:
if RUN_RAW_BACKGROUND_SCAN:
    print("Scanning pre-trigger region for uncorrelated pulses")
    by_key = {}
    for path in csv_files:
        m = FILENAME_RE.match(path.name)
        if m is None or m.group("channel") not in CHANNELS:
            continue
        by_key.setdefault((m.group("channel"), m.group("device")), []).append(path)

    scan_rows, noise_hist_rows, fit_rows = [], [], []
    pe_rows, staircase_rows, pe_estimate_rows = [], [], []
    for (channel, device), paths in sorted(by_key.items()):
        if MAX_FILES_PER_CHANNEL is not None:
            paths = paths[:MAX_FILES_PER_CHANNEL]
        counts = {s: 0 for s in SCAN_SIGMA_THRESHOLDS}
        live, peaks, noises, used = 0.0, [], [], 0
        hist_edges = np.linspace(-NOISE_HIST_RANGE_V, NOISE_HIST_RANGE_V, NOISE_HIST_BINS + 1)
        hist_counts = np.zeros(NOISE_HIST_BINS, dtype=np.int64)
        pe_edges = np.linspace(0.0, PE_HIST_MAX_V, PE_HIST_BINS + 1)
        pe_counts = np.zeros(PE_HIST_BINS, dtype=np.int64)
        for path in paths:
            try:
                frame = pd.read_csv(path, header=4, usecols=["Time", "Ampl"]).apply(
                    pd.to_numeric, errors="coerce").dropna()
            except Exception:
                continue
            t = frame["Time"].to_numpy(float)
            a = frame["Ampl"].to_numpy(float)
            m = (t >= PRETRIGGER_START_NS * 1e-9) & (t <= PRETRIGGER_END_NS * 1e-9)
            if np.count_nonzero(m) < 100:
                continue
            seg = a[m] - float(np.median(a[m]))
            noise = robust_sigma(seg)
            if not np.isfinite(noise) or noise <= 0:
                continue
            for s in SCAN_SIGMA_THRESHOLDS:
                counts[s] += n_excursions(seg, s * noise)
            hist_counts += np.histogram(seg, bins=hist_edges)[0]
            pe_counts += np.histogram(excursion_peaks(seg, PULSE_AMPLITUDE_MIN_SIGMA * noise),
                                      bins=pe_edges)[0]
            peaks.append(float(np.max(seg) / noise))
            noises.append(noise)
            live += float(t[m][-1] - t[m][0])
            used += 1
        if used == 0 or live <= 0:
            continue
        row = {"channel": channel, "device": device, "n_waveforms_scanned": used,
               "total_live_time_s": live, "live_time_per_waveform_ns": live / used * 1e9,
               "median_pretrigger_noise_v": float(np.median(noises)),
               "median_max_excursion_sigma": float(np.median(peaks)),
               "p99_max_excursion_sigma": float(np.percentile(peaks, 99))}
        for s in SCAN_SIGMA_THRESHOLDS:
            row[f"n_candidates_{s:g}sigma"] = counts[s]
            row[f"rate_hz_{s:g}sigma"] = counts[s] / live
        scan_rows.append(row)

        centers = 0.5 * (hist_edges[:-1] + hist_edges[1:])
        keep = hist_counts > 0
        noise_hist_rows.append(pd.DataFrame({
            "channel": channel, "device": device,
            "bin_center_v": centers[keep], "counts": hist_counts[keep]}))
        fit = fit_gaussian(centers, hist_counts)
        fit_rows.append({"channel": channel, "device": device,
                         "n_waveforms": used, "live_time_s": live, **fit})

        pe_centers = 0.5 * (pe_edges[:-1] + pe_edges[1:])
        pe_rows.append(pd.DataFrame({"channel": channel, "device": device,
                                     "amplitude_v": pe_centers, "counts": pe_counts}))

        cumulative = np.cumsum(pe_counts[::-1])[::-1]
        staircase_rows.append(pd.DataFrame({
            "channel": channel, "device": device, "threshold_v": pe_edges[:-1],
            "n_pulses_above": cumulative, "rate_hz": cumulative / live}))

        smooth = np.convolve(pe_counts.astype(float), np.ones(PE_SMOOTH_BINS) / PE_SMOOTH_BINS, mode="same")
        valid_bins = pe_centers >= PE_MIN_PEAK_V
        maxima = [i for i in range(1, len(smooth) - 1)
                  if valid_bins[i] and smooth[i] > smooth[i - 1] and smooth[i] >= smooth[i + 1]
                  and smooth[i] > 0.02 * smooth[valid_bins].max()]
        pe1 = float(pe_centers[maxima[0]]) if maxima else math.nan
        pe_row = {"channel": channel, "device": device, "live_time_s": live,
                  "n_waveforms": used, "pe1_amplitude_v": pe1,
                  "n_peaks_found": len(maxima),
                  "peak_positions_v": ";".join(f"{pe_centers[i]:.5f}" for i in maxima[:5])}
        for fraction, label in ((0.5, "dcr"), (1.5, "crosstalk"), (2.5, "pe25")):
            if np.isfinite(pe1):
                index = int(np.searchsorted(pe_edges[:-1], fraction * pe1, side="right") - 1)
                index = max(0, min(len(cumulative) - 1, index))
                pe_row[f"rate_hz_at_{fraction:g}pe"] = float(cumulative[index] / live)
            else:
                pe_row[f"rate_hz_at_{fraction:g}pe"] = math.nan
        pe_row["dark_count_rate_hz"] = pe_row["rate_hz_at_0.5pe"]
        pe_row["optical_crosstalk_probability"] = (
            pe_row["rate_hz_at_1.5pe"] / pe_row["rate_hz_at_0.5pe"]
            if pe_row["rate_hz_at_0.5pe"] > 0 else math.nan)
        pe_estimate_rows.append(pe_row)
        print(f"  scanned {device} {channel} ({used} waveforms)")

    raw_background = pd.DataFrame(scan_rows)
    write("raw_pretrigger_background.csv", raw_background)
    if noise_hist_rows:
        write("background_noise_histogram.csv", pd.concat(noise_hist_rows, ignore_index=True))
    if fit_rows:
        write("background_noise_gaussian_fit.csv", pd.DataFrame(fit_rows))
    if pe_rows:
        write("background_pulse_amplitude_histogram.csv", pd.concat(pe_rows, ignore_index=True))
        write("background_rate_vs_threshold.csv", pd.concat(staircase_rows, ignore_index=True))
        write("background_pe_estimate.csv", pd.DataFrame(pe_estimate_rows))

    analytic_rows = []
    for device, group in raw_background.groupby("device"):
        rates = group.set_index("channel")
        for s in SCAN_SIGMA_THRESHOLDS:
            for a, b in pairs:
                if a not in rates.index or b not in rates.index:
                    continue
                ra = float(rates.loc[a, f"rate_hz_{s:g}sigma"])
                rb = float(rates.loc[b, f"rate_hz_{s:g}sigma"])
                analytic_rows.append({"device": device, "channel_a": a, "channel_b": b,
                                      "threshold_sigma": s, "singles_rate_a_hz": ra,
                                      "singles_rate_b_hz": rb,
                                      "resolving_time_ns": ACCIDENTAL_RESOLVING_TIME_NS,
                                      "accidental_pair_rate_hz": 2.0 * ACCIDENTAL_RESOLVING_TIME_NS * 1e-9 * ra * rb})
    write("analytic_accidental_rates.csv", pd.DataFrame(analytic_rows))

Scanning pre-trigger region for uncorrelated pulses
  scanned hcsipm C1 (2296 waveforms)
  scanned sensor_brown_ch_J15 C1 (2 waveforms)
  scanned sipm C1 (288 waveforms)
  scanned hcsipm C2 (2296 waveforms)
  scanned sensor_brown_ch_J15 C2 (2 waveforms)
  scanned sipm C2 (288 waveforms)
  scanned hcsipm C3 (2296 waveforms)
  scanned sensor_brown_ch_J15 C3 (2 waveforms)
  scanned sipm C3 (288 waveforms)
  scanned hcsipm C4 (2296 waveforms)
  scanned sensor_brown_ch_J15 C4 (2 waveforms)
  scanned sipm C4 (288 waveforms)
  wrote raw_pretrigger_background.csv (12 rows)
  wrote background_noise_histogram.csv (13882 rows)
  wrote background_noise_gaussian_fit.csv (12 rows)
  wrote background_pulse_amplitude_histogram.csv (7200 rows)
  wrote background_rate_vs_threshold.csv (7200 rows)
  wrote background_pe_estimate.csv (12 rows)
  wrote analytic_accidental_rates.csv (72 rows)


In [13]:
config = [("input_directory", str(INPUT_DIR)), ("output_directory", str(OUTPUT_DIR)),
          ("search_start_ns", SEARCH_START_NS), ("search_end_ns", SEARCH_END_NS),
          ("require_cfd_in_window", REQUIRE_CFD_IN_WINDOW), ("max_fall_search_ns", MAX_FALL_SEARCH_NS),
          ("pileup_reject_ratio", PILEUP_REJECT_RATIO), ("pulse_count_sigma", PULSE_COUNT_SIGMA),
          ("baseline_start_ns", BASELINE_START_NS), ("baseline_end_ns", BASELINE_END_NS),
          ("min_baseline_points", MIN_BASELINE_POINTS), ("min_snr", MIN_SNR),
          ("leading_edge_sigma", LEADING_EDGE_SIGMA), ("reference_channel", REFERENCE_CHANNEL),
          ("coincidence_window_ns", COINCIDENCE_WINDOW_NS), ("outlier_sigma", OUTLIER_SIGMA),
          ("n_mixing_trials", N_MIXING_TRIALS), ("mixing_seed", MIXING_SEED),
          ("pretrigger_start_ns", PRETRIGGER_START_NS), ("pretrigger_end_ns", PRETRIGGER_END_NS),
          ("accidental_resolving_time_ns", ACCIDENTAL_RESOLVING_TIME_NS),
          ("exclude_from_timing", ";".join(EXCLUDE_FROM_TIMING)),
          ("fit_range_sigma", FIT_RANGE_SIGMA), ("noise_hist_bins", NOISE_HIST_BINS),
          ("noise_hist_range_v", NOISE_HIST_RANGE_V), ("delta_t_fit_bins", DELTA_T_FIT_BINS)]
config += [(f"run:{r}:{c}", d) for r, m in runs.items() for c, d in m.items()]
write("analysis_configuration.csv", pd.DataFrame(config, columns=["parameter", "value"]))

write("csv_output_manifest.csv", pd.DataFrame(
    [{"csv_file": n, "rows": r, "path": str((OUTPUT_DIR / n).resolve())} for n, r in sorted(written.items())]))

print(f"\nDone. {len(written)} CSV files in {OUTPUT_DIR}")

  wrote analysis_configuration.csv (34 rows)
  wrote csv_output_manifest.csv (31 rows)

Done. 32 CSV files in C:\Users\firep\Downloads\simple_analysis_script\timing_results_v2
